# Experiment C: Pi-Ratings GLM

Drop-in replacement of ELO features with Pi-Ratings in the Poisson GLM.

**Hypothesis:** Pi-ratings encode score-margin (goal difference) and separate home/away strength,
discarded by ELO. This should produce a stronger signal for Poisson goal prediction.

**Setup:**
- Replace `home_elo`, `away_elo`, `elo_diff` with `pi_home`, `pi_away`, `pi_diff`
- Keep all form features identical to Notebook 11 baseline
- Same 4-fold LOTO-CV over WC 2010/2014/2018/2022
- Pi-ratings computed from ALL available history, but only matches BEFORE the WC year (no leakage)

**Baseline:** Poisson GLM with ELO → 4.336 pts/match (Notebook 11)

In [1]:
import sys
sys.path.insert(0, '../../src')

import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
from functools import lru_cache
from scipy.stats import poisson
from sklearn.linear_model import PoissonRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from pathlib import Path

from penaltyblog.ratings import PiRatingSystem

from evaluation.sporza import score_breakdown, sporza_points_series

DATA = Path('../../data/processed')
INTERIM = Path('../../data/interim')
WC_YEARS = [2010, 2014, 2018, 2022]
MAX_GOALS = 8

# Pi-rating GLM features: same as ELO version but with pi_ instead of elo_
HOME_FEATURES = ['pi_home', 'pi_away', 'pi_diff', 'is_neutral',
                 'home_form_wr', 'away_form_wr', 'home_form_gf', 'away_form_gf',
                 'home_form_ga', 'away_form_ga', 'tournament_weight']
AWAY_FEATURES = ['pi_away', 'pi_home', 'pi_diff', 'is_neutral',
                 'away_form_wr', 'home_form_wr', 'away_form_gf', 'home_form_gf',
                 'away_form_ga', 'home_form_ga', 'tournament_weight']

print('Feature sets defined.')
print('HOME_FEATURES:', HOME_FEATURES)
print('AWAY_FEATURES:', AWAY_FEATURES)

Feature sets defined.
HOME_FEATURES: ['pi_home', 'pi_away', 'pi_diff', 'is_neutral', 'home_form_wr', 'away_form_wr', 'home_form_gf', 'away_form_gf', 'home_form_ga', 'away_form_ga', 'tournament_weight']
AWAY_FEATURES: ['pi_away', 'pi_home', 'pi_diff', 'is_neutral', 'away_form_wr', 'home_form_wr', 'away_form_gf', 'home_form_gf', 'away_form_ga', 'home_form_ga', 'tournament_weight']


In [2]:
def bootstrap_ci(pts: pd.Series, n_boot: int = 10000) -> dict:
    rng = np.random.default_rng(42)
    vals = pts.values
    boot = np.array([rng.choice(vals, size=len(vals), replace=True).mean() for _ in range(n_boot)])
    lo, hi = np.percentile(boot, [2.5, 97.5])
    return {'mean': float(vals.mean()), 'ci_lo': float(lo), 'ci_hi': float(hi)}


def permutation_test(pts_a: np.ndarray, pts_b: np.ndarray, n_perm: int = 10000) -> float:
    """Paired permutation test: H0 = mean(a) <= mean(b). Returns p-value."""
    rng = np.random.default_rng(42)
    obs_diff = pts_a.mean() - pts_b.mean()
    diffs = np.zeros(n_perm)
    for i in range(n_perm):
        swap = rng.integers(0, 2, size=len(pts_a)).astype(bool)
        a = np.where(swap, pts_b, pts_a)
        b = np.where(swap, pts_a, pts_b)
        diffs[i] = a.mean() - b.mean()
    return float((diffs >= obs_diff).mean())


def build_X(df: pd.DataFrame, features: list) -> np.ndarray:
    X = df[features].copy().fillna(df[features].median())
    return X.values


def build_pipe() -> Pipeline:
    return Pipeline([('scaler', StandardScaler()),
                     ('poisson', PoissonRegressor(alpha=0.1, max_iter=300))])


@lru_cache(maxsize=50000)
def best_score_cached(lh_r: float, la_r: float) -> tuple:
    """Score (h, a) maximising expected Sporza pts given Poisson(lh) x Poisson(la)."""
    goals = np.arange(MAX_GOALS + 1)
    ph_pmf = poisson.pmf(goals, lh_r)
    pa_pmf = poisson.pmf(goals, la_r)
    joint = np.outer(ph_pmf, pa_pmf)

    def sporza_mat(pred_h: int, pred_a: int) -> float:
        pts = 0.0
        for ah in range(MAX_GOALS + 1):
            for aa in range(MAX_GOALS + 1):
                p = joint[ah, aa]
                if p < 1e-9:
                    continue
                if pred_h == ah and pred_a == aa:
                    pts += p * 10
                elif (pred_h - pred_a) == (ah - aa):
                    pts += p * 7
                elif np.sign(pred_h - pred_a) == np.sign(ah - aa):
                    pts += p * 5
                else:
                    pts += p * 1
        return pts

    best_pts, best_h, best_a = -1.0, 1, 0
    for ph_idx in range(MAX_GOALS + 1):
        for pa_idx in range(MAX_GOALS + 1):
            ep = sporza_mat(ph_idx, pa_idx)
            if ep > best_pts:
                best_pts, best_h, best_a = ep, ph_idx, pa_idx
    return best_h, best_a

print('Helper functions defined.')

Helper functions defined.


In [3]:
def compute_pi_ratings_before_cutoff(matches: pd.DataFrame, cutoff_date: str) -> dict:
    """
    Compute pi-ratings using all matches strictly before `cutoff_date`.
    Returns dict: {team: {'home': float, 'away': float}}
    
    Ratings are processed chronologically so that each match update uses
    only prior-match information.
    """
    pi = PiRatingSystem(alpha=0.15, beta=0.10, k=0.75)
    training_matches = matches[matches['date'] < cutoff_date].sort_values('date')
    
    for _, row in training_matches.iterrows():
        goal_diff = int(row['home_score'] - row['away_score'])
        pi.update_ratings(row['home_team'], row['away_team'], goal_diff)
    
    print(f'  Cutoff {cutoff_date}: processed {len(training_matches)} matches, '
          f'{len(pi.team_ratings)} teams rated')
    return pi.team_ratings


def add_pi_features(df: pd.DataFrame, team_ratings: dict) -> pd.DataFrame:
    """
    Add pi_home, pi_away, pi_diff to a match DataFrame.
    
    For each match, pi_home is the home team's HOME rating (their attacking home strength),
    pi_away is the away team's AWAY rating (their defending away strength).
    pi_diff = pi_home_rating - pi_away_rating (expected goal difference signal).
    """
    df = df.copy()
    
    def get_home_rating(team):
        if team in team_ratings:
            return team_ratings[team]['home']
        return 0.0  # default for unseen teams
    
    def get_away_rating(team):
        if team in team_ratings:
            return team_ratings[team]['away']
        return 0.0  # default for unseen teams
    
    df['pi_home'] = df['home_team'].map(get_home_rating)
    df['pi_away'] = df['away_team'].map(get_away_rating)
    # pi_diff = expected goal difference: home_home_rating - away_away_rating
    df['pi_diff'] = df['pi_home'] - df['pi_away']
    
    return df


# Load the full match history for pi-rating computation
matches_full = pd.read_parquet(INTERIM / 'matches.parquet')
print(f'Full match history: {len(matches_full)} matches ({matches_full.date.min().date()} to {matches_full.date.max().date()})')

# Load autofill baseline
import json
manifest = json.loads((DATA / 'split_manifest.json').read_text())
autofill_mean = manifest['autofill_baseline']['pooled_mean_pts']
print(f'Autofill baseline: {autofill_mean:.3f} pts/match')

Full match history: 32288 matches (1990-01-12 to 2026-06-11)
Autofill baseline: 4.137 pts/match


In [4]:
# Pre-compute pi-ratings for each WC year cutoff (no leakage)
WC_CUTOFFS = {
    2010: '2010-06-01',
    2014: '2014-06-01',
    2018: '2018-06-01',
    2022: '2022-11-01',
}

print('Computing pi-ratings for each fold cutoff...')
pi_ratings_by_year = {}
for year, cutoff in WC_CUTOFFS.items():
    pi_ratings_by_year[year] = compute_pi_ratings_before_cutoff(matches_full, cutoff)

print('\nPi-rating value ranges by fold:')
for year, ratings in pi_ratings_by_year.items():
    home_vals = [v['home'] for v in ratings.values()]
    away_vals = [v['away'] for v in ratings.values()]
    print(f'  WC {year}: home [{min(home_vals):.3f}, {max(home_vals):.3f}], '
          f'away [{min(away_vals):.3f}, {max(away_vals):.3f}]')

Computing pi-ratings for each fold cutoff...


  Cutoff 2010-06-01: processed 16704 matches, 275 teams rated


  Cutoff 2014-06-01: processed 20658 matches, 294 teams rated


  Cutoff 2018-06-01: processed 24399 matches, 312 teams rated


  Cutoff 2022-11-01: processed 28505 matches, 323 teams rated

Pi-rating value ranges by fold:
  WC 2010: home [-3.654, 2.824], away [-4.415, 2.548]
  WC 2014: home [-3.404, 3.302], away [-4.005, 3.100]
  WC 2018: home [-3.428, 3.352], away [-4.521, 3.401]
  WC 2022: home [-3.507, 3.594], away [-4.740, 3.718]


In [5]:
# Sanity check: compare well-known teams at WC 2022 cutoff
ratings_2022 = pi_ratings_by_year[2022]
top_teams = ['Brazil', 'France', 'Germany', 'Argentina', 'England', 'Spain', 'Netherlands']
print('Pi-ratings at WC 2022 cutoff (home | away | avg):')
for team in top_teams:
    if team in ratings_2022:
        h = ratings_2022[team]['home']
        a = ratings_2022[team]['away']
        print(f'  {team:20s}: home={h:+.3f}  away={a:+.3f}  avg={((h+a)/2):+.3f}')
    else:
        print(f'  {team:20s}: NOT FOUND')

Pi-ratings at WC 2022 cutoff (home | away | avg):
  Brazil              : home=+3.594  away=+3.718  avg=+3.656
  France              : home=+2.610  away=+2.316  avg=+2.463
  Germany             : home=+2.711  away=+2.471  avg=+2.591
  Argentina           : home=+3.388  away=+3.079  avg=+3.233
  England             : home=+2.681  away=+2.503  avg=+2.592
  Spain               : home=+2.989  away=+2.347  avg=+2.668
  Netherlands         : home=+2.847  away=+2.494  avg=+2.671


In [6]:
# LOTO-CV with Pi-Ratings GLM
all_pts = []
fold_results = []
fold_pipes = {}

for year in WC_YEARS:
    train = pd.read_parquet(DATA / f'train_fold_{year}.parquet')
    evalf = pd.read_parquet(DATA / f'eval_fold_{year}.parquet')
    
    # Add pi-rating features using cutoff ratings (before the WC year)
    ratings = pi_ratings_by_year[year]
    train = add_pi_features(train, ratings)
    evalf = add_pi_features(evalf, ratings)
    
    # Check coverage: how many teams have pi-ratings vs defaulting to 0.0
    train_missing = train['home_team'].apply(lambda t: t not in ratings).sum() + \
                    train['away_team'].apply(lambda t: t not in ratings).sum()
    eval_missing = evalf['home_team'].apply(lambda t: t not in ratings).sum() + \
                   evalf['away_team'].apply(lambda t: t not in ratings).sum()
    print(f'WC {year}: train missing {train_missing}/{len(train)*2} teams, '
          f'eval missing {eval_missing}/{len(evalf)*2} teams')
    
    # Stacked training: each match as (home perspective) + (away perspective)
    X_h = build_X(train, HOME_FEATURES)
    X_a = build_X(train, AWAY_FEATURES)
    X_tr = np.vstack([X_h, X_a])
    y_tr = np.concatenate([train['home_score'].values, train['away_score'].values])
    
    pipe = build_pipe()
    pipe.fit(X_tr, y_tr)
    fold_pipes[year] = pipe
    
    lh = pipe.predict(build_X(evalf, HOME_FEATURES))
    la = pipe.predict(build_X(evalf, AWAY_FEATURES))
    
    preds = [best_score_cached(round(float(h), 2), round(float(a), 2)) for h, a in zip(lh, la)]
    pred_home = pd.Series([p[0] for p in preds])
    pred_away = pd.Series([p[1] for p in preds])
    
    bd = score_breakdown(pred_home, pred_away,
                         evalf['home_score'].reset_index(drop=True),
                         evalf['away_score'].reset_index(drop=True))
    pts = sporza_points_series(pred_home, pred_away,
                               evalf['home_score'].reset_index(drop=True),
                               evalf['away_score'].reset_index(drop=True))
    all_pts.extend(pts.tolist())
    fold_results.append({'year': year, 'mean_pts': bd['mean_pts'],
                         'pct_exact': bd['pct_exact'], 'pct_correct_result': bd['pct_correct_result']})
    print(f'  -> WC {year}: mean_pts={bd["mean_pts"]:.3f}  exact={bd["pct_exact"]:.1%}  correct={bd["pct_correct_result"]:.1%}\n')

pooled_pts = pd.Series(all_pts)
ci = bootstrap_ci(pooled_pts)
print(f'Pooled LOTO-CV ({len(pooled_pts)} matches): {ci["mean"]:.3f} pts/match  '
      f'95% CI [{ci["ci_lo"]:.3f}, {ci["ci_hi"]:.3f}]  '
      f'CI width={ci["ci_hi"]-ci["ci_lo"]:.3f}')

WC 2010: train missing 0/438 teams, eval missing 0/128 teams
  -> WC 2010: mean_pts=4.266  exact=20.3%  correct=51.6%

WC 2014: train missing 0/7800 teams, eval missing 0/128 teams
  -> WC 2014: mean_pts=4.969  exact=18.8%  correct=64.1%



WC 2018: train missing 0/14544 teams, eval missing 0/128 teams
  -> WC 2018: mean_pts=4.797  exact=20.3%  correct=60.9%

WC 2022: train missing 0/22212 teams, eval missing 0/128 teams
  -> WC 2022: mean_pts=3.469  exact=6.2%  correct=50.0%



Pooled LOTO-CV (256 matches): 4.375 pts/match  95% CI [3.977, 4.781]  CI width=0.805


In [7]:
# Comparison vs baselines
ELO_GLM_MEAN = 4.336  # Notebook 11 baseline

print('Per-fold results:')
for r in fold_results:
    print(f'  WC {r["year"]}: {r["mean_pts"]:.3f} pts  exact={r["pct_exact"]:.1%}  correct={r["pct_correct_result"]:.1%}')

print(f'\nComparison vs baselines:')
print(f'  Autofill baseline:        {autofill_mean:.3f} pts/match')
print(f'  ELO Poisson GLM (nb 11):  {ELO_GLM_MEAN:.3f} pts/match')
print(f'  Pi-Ratings GLM (exp C):   {ci["mean"]:.3f} pts/match  95% CI [{ci["ci_lo"]:.3f}, {ci["ci_hi"]:.3f}]')
print(f'  Delta vs ELO GLM:         {ci["mean"] - ELO_GLM_MEAN:+.3f} pts/match')
print(f'  Delta vs autofill:        {ci["mean"] - autofill_mean:+.3f} pts/match')

beats_elo = ci['ci_lo'] > ELO_GLM_MEAN
beats_autofill = ci['ci_lo'] > autofill_mean
print(f'\n  CI lower bound > ELO GLM:   {beats_elo} → {"statistically beats" if beats_elo else "not conclusive"}')
print(f'  CI lower bound > autofill:  {beats_autofill} → {"statistically beats" if beats_autofill else "not conclusive"}')

Per-fold results:
  WC 2010: 4.266 pts  exact=20.3%  correct=51.6%
  WC 2014: 4.969 pts  exact=18.8%  correct=64.1%
  WC 2018: 4.797 pts  exact=20.3%  correct=60.9%
  WC 2022: 3.469 pts  exact=6.2%  correct=50.0%

Comparison vs baselines:
  Autofill baseline:        4.137 pts/match
  ELO Poisson GLM (nb 11):  4.336 pts/match
  Pi-Ratings GLM (exp C):   4.375 pts/match  95% CI [3.977, 4.781]
  Delta vs ELO GLM:         +0.039 pts/match
  Delta vs autofill:        +0.238 pts/match

  CI lower bound > ELO GLM:   False → not conclusive
  CI lower bound > autofill:  False → not conclusive


In [8]:
# Paired permutation test vs ELO GLM (requires ELO pts series from notebook 11)
# Re-run ELO GLM to get per-match pts for paired test
from sklearn.linear_model import PoissonRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

ELO_HOME_FEATURES = ['home_elo', 'away_elo', 'elo_diff', 'is_neutral',
                     'home_form_wr', 'away_form_wr', 'home_form_gf', 'away_form_gf',
                     'home_form_ga', 'away_form_ga', 'tournament_weight']
ELO_AWAY_FEATURES = ['away_elo', 'home_elo', 'elo_diff', 'is_neutral',
                     'away_form_wr', 'home_form_wr', 'away_form_gf', 'home_form_gf',
                     'away_form_ga', 'home_form_ga', 'tournament_weight']

elo_all_pts = []
for year in WC_YEARS:
    train = pd.read_parquet(DATA / f'train_fold_{year}.parquet')
    evalf = pd.read_parquet(DATA / f'eval_fold_{year}.parquet')
    
    X_h = build_X(train, ELO_HOME_FEATURES)
    X_a = build_X(train, ELO_AWAY_FEATURES)
    X_tr = np.vstack([X_h, X_a])
    y_tr = np.concatenate([train['home_score'].values, train['away_score'].values])
    
    pipe_elo = build_pipe()
    pipe_elo.fit(X_tr, y_tr)
    
    lh = pipe_elo.predict(build_X(evalf, ELO_HOME_FEATURES))
    la = pipe_elo.predict(build_X(evalf, ELO_AWAY_FEATURES))
    
    preds = [best_score_cached(round(float(h), 2), round(float(a), 2)) for h, a in zip(lh, la)]
    pred_home = pd.Series([p[0] for p in preds])
    pred_away = pd.Series([p[1] for p in preds])
    
    pts_elo = sporza_points_series(pred_home, pred_away,
                                   evalf['home_score'].reset_index(drop=True),
                                   evalf['away_score'].reset_index(drop=True))
    elo_all_pts.extend(pts_elo.tolist())

elo_pts_series = np.array(elo_all_pts)
pi_pts_series = np.array(all_pts)

p_val = permutation_test(pi_pts_series, elo_pts_series)
print(f'Paired permutation test (Pi-Ratings > ELO GLM): p = {p_val:.4f}')
print(f'(p < 0.05 means Pi-Ratings is significantly better than ELO GLM)')

Paired permutation test (Pi-Ratings > ELO GLM): p = 0.3980
(p < 0.05 means Pi-Ratings is significantly better than ELO GLM)


In [9]:
# Log to MLflow
mlflow.set_tracking_uri('sqlite:////Users/seppe.vanswegenoven/projects/wk_2026_match_predictor/mlflow.db')
mlflow.set_experiment('wk2026-tabular-2026-06-14')

with mlflow.start_run(run_name='pi_ratings_glm_loto') as run:
    mlflow.set_tags({
        'modality': 'tabular',
        'approach': 'poisson_glm_pi_ratings',
        'eval_strategy': 'loto_cv',
        'dataset_version': 'split_v2',
        'experiment': 'C',
        'hypothesis': 'pi_ratings_replace_elo'
    })
    mlflow.log_params({
        'features': ','.join(HOME_FEATURES),
        'alpha': 0.1,
        'score_strategy': 'max_expected_sporza_pts',
        'max_goals_grid': MAX_GOALS,
        'pi_alpha': 0.15,
        'pi_beta': 0.10,
        'pi_k': 0.75,
        'elo_glm_baseline': ELO_GLM_MEAN,
    })
    mlflow.log_metric('loto_mean_sporza_pts', ci['mean'])
    mlflow.log_metric('loto_ci_lo', ci['ci_lo'])
    mlflow.log_metric('loto_ci_hi', ci['ci_hi'])
    mlflow.log_metric('autofill_baseline', autofill_mean)
    mlflow.log_metric('delta_vs_autofill', ci['mean'] - autofill_mean)
    mlflow.log_metric('delta_vs_elo_glm', ci['mean'] - ELO_GLM_MEAN)
    mlflow.log_metric('perm_test_p_vs_elo_glm', p_val)
    for r in fold_results:
        mlflow.log_metric(f'fold_{r["year"]}_mean_pts', r['mean_pts'])
        mlflow.log_metric(f'fold_{r["year"]}_pct_exact', r['pct_exact'])
        mlflow.log_metric(f'fold_{r["year"]}_pct_correct', r['pct_correct_result'])
    mlflow.sklearn.log_model(fold_pipes[2022], 'pi_glm_wc2022_fold')
    run_id = run.info.run_id

print(f'MLflow run_id: {run_id}')
print(f'\nFinal summary:')
print(f'  Pi-Ratings GLM:  {ci["mean"]:.3f} pts/match  95% CI [{ci["ci_lo"]:.3f}, {ci["ci_hi"]:.3f}]')
print(f'  Delta vs ELO:    {ci["mean"] - ELO_GLM_MEAN:+.3f} pts/match')

2026/06/14 17:38:10 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


2026/06/14 17:38:11 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


2026/06/14 17:38:12 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


MLflow run_id: dd71d64213b24f799f5388bce8bc7587

Final summary:
  Pi-Ratings GLM:  4.375 pts/match  95% CI [3.977, 4.781]
  Delta vs ELO:    +0.039 pts/match
